<a href="https://colab.research.google.com/github/Althaf12344/Ai-Skill-Project-VU/blob/main/CLA_1_MLOPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "feast>=0.60,<0.61" pandas numpy scikit-learn pyarrow openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving AI_Skill_Gap_Classification_Dataset_500.xlsx to AI_Skill_Gap_Classification_Dataset_500.xlsx


In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
import numpy as np

from feast import FeatureStore

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Create project structure

PROJECT = Path("/content/skill_gap_feast")

DATA_DIR = PROJECT / "data"
FEATURE_REPO = PROJECT / "feature_repo"
RESULTS_DIR = PROJECT / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_REPO.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project created at:", PROJECT)

Project created at: /content/skill_gap_feast


In [ ]:
import shutil
from pathlib import Path

uploaded_file = list(uploaded.keys())[0]

source_file = Path("/content") / uploaded_file
destination_file = DATA_DIR / "skill_gap_dataset.xlsx"

shutil.copy(source_file, destination_file)

print("Dataset copied to:")
print(destination_file)

Dataset copied to:
/content/skill_gap_feast/data/skill_gap_dataset.xlsx


In [ ]:
df = pd.read_excel(destination_file)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (500, 16)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Student_ID,Age,Gender,Education_Level,Programming_Skill,Python_Experience_Years,Math_Skill,ML_Knowledge,AI_Project_Count,Online_Courses_Completed,Coding_Hours_Per_Week,Communication_Skill,Problem_Solving,AI_Certification,Internship,Skill_Gap
0,1,28,Male,Diploma,7,2,9,8,5,7,38,7,3,Yes,Yes,Low
1,2,21,Female,UG,7,2,8,8,8,14,15,7,7,Yes,No,Low
2,3,19,Male,PG,7,3,10,9,6,7,21,4,8,Yes,Yes,Low
3,4,26,Female,Diploma,9,4,7,10,8,6,27,8,8,Yes,Yes,Low
4,5,19,Female,PG,8,2,7,8,10,9,17,7,8,Yes,Yes,Low


In [ ]:
print("Columns:")

for column in df.columns:
    print("-", column)

Columns:
- Student_ID
- Age
- Gender
- Education_Level
- Programming_Skill
- Python_Experience_Years
- Math_Skill
- ML_Knowledge
- AI_Project_Count
- Online_Courses_Completed
- Coding_Hours_Per_Week
- Communication_Skill
- Problem_Solving
- AI_Certification
- Internship
- Skill_Gap


In [ ]:
features = pd.DataFrame()

# ------------------------------------------------
# Entity
# ------------------------------------------------

features["student_id"] = df["Student_ID"].astype("int64")


# ------------------------------------------------
# Feast timestamps
# ------------------------------------------------

features["event_timestamp"] = (
    pd.Timestamp("2025-01-01", tz="UTC")
    + pd.to_timedelta(np.arange(len(df)), unit="D")
)

features["created_timestamp"] = features["event_timestamp"]


# ------------------------------------------------
# Raw numerical features
# ------------------------------------------------

features["age"] = df["Age"].astype("int64")

features["programming_skill"] = (
    df["Programming_Skill"].astype("int64")
)

features["python_experience_years"] = (
    df["Python_Experience_Years"].astype("int64")
)

features["math_skill"] = (
    df["Math_Skill"].astype("int64")
)

features["ml_knowledge"] = (
    df["ML_Knowledge"].astype("int64")
)

features["ai_project_count"] = (
    df["AI_Project_Count"].astype("int64")
)

features["online_courses_completed"] = (
    df["Online_Courses_Completed"].astype("int64")
)

features["coding_hours_per_week"] = (
    df["Coding_Hours_Per_Week"].astype("int64")
)

features["communication_skill"] = (
    df["Communication_Skill"].astype("int64")
)

features["problem_solving"] = (
    df["Problem_Solving"].astype("int64")
)


# ------------------------------------------------
# Categorical → numerical
# ------------------------------------------------

features["certification_flag"] = (
    df["AI_Certification"]
    .map({"Yes": 1, "No": 0})
    .astype("int64")
)

features["internship_flag"] = (
    df["Internship"]
    .map({"Yes": 1, "No": 0})
    .astype("int64")
)


# ------------------------------------------------
# FEATURE 1
# Technical Skill Score
# ------------------------------------------------

features["technical_skill_score"] = (
    df[
        [
            "Programming_Skill",
            "Math_Skill",
            "ML_Knowledge"
        ]
    ].mean(axis=1)
).astype("float32")


# ------------------------------------------------
# FEATURE 2
# Experience Score
# ------------------------------------------------

features["experience_score"] = (
    (
        (df["Python_Experience_Years"] / 5.0 * 10.0)
        +
        (df["AI_Project_Count"] / 10.0 * 10.0)
    ) / 2.0
).astype("float32")


# ------------------------------------------------
# FEATURE 3
# Learning Engagement Score
# ------------------------------------------------

features["learning_engagement_score"] = (
    (
        (df["Online_Courses_Completed"] / 15.0 * 10.0)
        +
        (df["Coding_Hours_Per_Week"] / 40.0 * 10.0)
    ) / 2.0
).astype("float32")


# ------------------------------------------------
# FEATURE 4
# Soft Skill Score
# ------------------------------------------------

features["soft_skill_score"] = (
    df[
        [
            "Communication_Skill",
            "Problem_Solving"
        ]
    ].mean(axis=1)
).astype("float32")


# ------------------------------------------------
# FEATURE 5
# Industry Readiness Score
# ------------------------------------------------

features["industry_readiness_score"] = (
    0.35 * features["technical_skill_score"]
    +
    0.25 * features["experience_score"]
    +
    0.20 * features["learning_engagement_score"]
    +
    0.20 * features["soft_skill_score"]
).astype("float32")


# ------------------------------------------------
# Target
# ------------------------------------------------

features["target"] = (
    df["Skill_Gap"]
    .map({
        "Low": 0,
        "Medium": 1,
        "High": 2
    })
    .astype("int64")
)

features["target_label"] = df["Skill_Gap"]


display(features.head())

,student_id,event_timestamp,created_timestamp,age,programming_skill,python_experience_years,math_skill,ml_knowledge,ai_project_count,online_courses_completed,...,problem_solving,certification_flag,internship_flag,technical_skill_score,experience_score,learning_engagement_score,soft_skill_score,industry_readiness_score,target,target_label
0,1,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,28,7,2,9,8,5,7,...,3,1,1,8.000000,4.5,7.083333,5.0,6.341667,0,Low
1,2,2025-01-02 00:00:00+00:00,2025-01-02 00:00:00+00:00,21,7,2,8,8,8,14,...,7,1,0,7.666667,6.0,6.541667,7.0,6.891667,0,Low
2,3,2025-01-03 00:00:00+00:00,2025-01-03 00:00:00+00:00,19,7,3,10,9,6,7,...,8,1,1,8.666667,6.0,4.958333,6.0,6.725000,0,Low
3,4,2025-01-04 00:00:00+00:00,2025-01-04 00:00:00+00:00,26,9,4,7,10,8,6,...,8,1,1,8.666667,8.0,5.375000,8.0,7.708333,0,Low
4,5,2025-01-05 00:00:00+00:00,2025-01-05 00:00:00+00:00,19,8,2,7,8,10,9,...,8,1,1,7.666667,7.0,5.125000,7.5,6.958333,0,Low


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
PARQUET_FILE = DATA_DIR / "skill_features.parquet"

features.to_parquet(
    PARQUET_FILE,
    index=False
)

print("Parquet file created:")
print(PARQUET_FILE)

Parquet file created:
/content/skill_gap_feast/data/skill_features.parquet


In [ ]:
print("Original dataset shape:")
print(df.shape)

print()

print("Feature dataset shape:")
print(features.shape)

print()

print("Feature columns:")
for col in features.columns:
    print("-", col)

Original dataset shape:
(500, 16)

Feature dataset shape:
(500, 22)

Feature columns:
- student_id
- event_timestamp
- created_timestamp
- age
- programming_skill
- python_experience_years
- math_skill
- ml_knowledge
- ai_project_count
- online_courses_completed
- coding_hours_per_week
- communication_skill
- problem_solving
- certification_flag
- internship_flag
- technical_skill_score
- experience_score
- learning_engagement_score
- soft_skill_score
- industry_readiness_score
- target
- target_label


In [ ]:
feature_store_yaml = """
project: skill_gap_feast

registry: data/registry.db

provider: local

online_store:
  type: sqlite
  path: data/online_store.db

entity_key_serialization_version: 3
"""

(FEATURE_REPO / "feature_store.yaml").write_text(
    feature_store_yaml
)

print(feature_store_yaml)


project: skill_gap_feast

registry: data/registry.db

provider: local

online_store:
  type: sqlite
  path: data/online_store.db

entity_key_serialization_version: 3



In [ ]:
features_py = '''
from datetime import timedelta

from feast import Entity
from feast import FeatureView
from feast import Field
from feast import FileSource

from feast.types import Int64
from feast.types import Float32


# ==========================================
# ENTITY
# ==========================================

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="Student identifier"
)


# ==========================================
# DATA SOURCE
# ==========================================

skill_gap_source = FileSource(
    name="skill_gap_source",
    path="../data/skill_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)


# ==========================================
# FEATURE VIEW
# ==========================================

skill_gap_features = FeatureView(
    name="skill_gap_features",

    entities=[student],

    ttl=timedelta(days=3650),

    schema=[

        Field(
            name="age",
            dtype=Int64
        ),

        Field(
            name="programming_skill",
            dtype=Int64
        ),

        Field(
            name="python_experience_years",
            dtype=Int64
        ),

        Field(
            name="math_skill",
            dtype=Int64
        ),

        Field(
            name="ml_knowledge",
            dtype=Int64
        ),

        Field(
            name="ai_project_count",
            dtype=Int64
        ),

        Field(
            name="online_courses_completed",
            dtype=Int64
        ),

        Field(
            name="coding_hours_per_week",
            dtype=Int64
        ),

        Field(
            name="communication_skill",
            dtype=Int64
        ),

        Field(
            name="problem_solving",
            dtype=Int64
        ),

        Field(
            name="certification_flag",
            dtype=Int64
        ),

        Field(
            name="internship_flag",
            dtype=Int64
        ),

        Field(
            name="technical_skill_score",
            dtype=Float32
        ),

        Field(
            name="experience_score",
            dtype=Float32
        ),

        Field(
            name="learning_engagement_score",
            dtype=Float32
        ),

        Field(
            name="soft_skill_score",
            dtype=Float32
        ),

        Field(
            name="industry_readiness_score",
            dtype=Float32
        )
    ],

    online=True,

    source=skill_gap_source,

    tags={
        "domain": "curriculum-industry-skill-gap"
    }
)
'''

(FEATURE_REPO / "features.py").write_text(features_py)

print(features_py)


from datetime import timedelta

from feast import Entity
from feast import FeatureView
from feast import Field
from feast import FileSource

from feast.types import Int64
from feast.types import Float32


# ==========================================
# ENTITY
# ==========================================

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="Student identifier"
)


# ==========================================
# DATA SOURCE
# ==========================================

skill_gap_source = FileSource(
    name="skill_gap_source",
    path="../data/skill_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)


# ==========================================
# FEATURE VIEW
# ==========================================

skill_gap_features = FeatureView(
    name="skill_gap_features",

    entities=[student],

    ttl=timedelta(days=3650),

    schema=[

        Field(
            name="age",
    

In [ ]:
!ls -R /content/skill_gap_feast

/content/skill_gap_feast:
data  feature_repo  results

/content/skill_gap_feast/data:
skill_features.parquet	skill_gap_dataset.xlsx

/content/skill_gap_feast/feature_repo:
features.py  feature_store.yaml

/content/skill_gap_feast/results:


In [ ]:
%cd /content/skill_gap_feast

/content/skill_gap_feast


In [ ]:
%cd /content/skill_gap_feast/feature_repo

!feast apply

/content/skill_gap_feast/feature_repo
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path=str(FEATURE_REPO)
)

print("Entities:")
for entity in store.list_entities():
    print(entity.name)

print("\nFeature Views:")
for fv in store.list_all_feature_views():
    print(fv.name)

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path=str(FEATURE_REPO)
)

print("Entities:")
for entity in store.list_entities():
    print(entity.name)

print("\nFeature Views:")
for fv in store.list_all_feature_views():
    print(fv.name)

Entities:
student

Feature Views:
skill_gap_features


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
entity_df = features[
    [
        "student_id",
        "event_timestamp",
        "target"
    ]
].copy()

display(entity_df.head())

,student_id,event_timestamp,target
0,1,2025-01-01 00:00:00+00:00,0
1,2,2025-01-02 00:00:00+00:00,0
2,3,2025-01-03 00:00:00+00:00,0
3,4,2025-01-04 00:00:00+00:00,0
4,5,2025-01-05 00:00:00+00:00,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_refs = [
    "skill_gap_features:age",
    "skill_gap_features:programming_skill",
    "skill_gap_features:python_experience_years",
    "skill_gap_features:math_skill",
    "skill_gap_features:ml_knowledge",
    "skill_gap_features:ai_project_count",
    "skill_gap_features:online_courses_completed",
    "skill_gap_features:coding_hours_per_week",
    "skill_gap_features:communication_skill",
    "skill_gap_features:problem_solving",
    "skill_gap_features:certification_flag",
    "skill_gap_features:internship_flag",
    "skill_gap_features:technical_skill_score",
    "skill_gap_features:experience_score",
    "skill_gap_features:learning_engagement_score",
    "skill_gap_features:soft_skill_score",
    "skill_gap_features:industry_readiness_score"
]

print("Number of Feast features:", len(feature_refs))

Number of Feast features: 17


In [ ]:
historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs
).to_df()

display(historical_features.head())

,student_id,event_timestamp,target,age,programming_skill,python_experience_years,math_skill,ml_knowledge,ai_project_count,online_courses_completed,coding_hours_per_week,communication_skill,problem_solving,certification_flag,internship_flag,technical_skill_score,experience_score,learning_engagement_score,soft_skill_score,industry_readiness_score
0,1,2025-01-01 00:00:00+00:00,0,28,7,2,9,8,5,7,38,7,3,1,1,8.000000,4.5,7.083333,5.0,6.341667
1,2,2025-01-02 00:00:00+00:00,0,21,7,2,8,8,8,14,15,7,7,1,0,7.666667,6.0,6.541667,7.0,6.891667
2,3,2025-01-03 00:00:00+00:00,0,19,7,3,10,9,6,7,21,4,8,1,1,8.666667,6.0,4.958333,6.0,6.725000
3,4,2025-01-04 00:00:00+00:00,0,26,9,4,7,10,8,6,27,8,8,1,1,8.666667,8.0,5.375000,8.0,7.708333
4,5,2025-01-05 00:00:00+00:00,0,19,8,2,7,8,10,9,17,7,8,1,1,7.666667,7.0,5.125000,7.5,6.958333


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
store.get_historical_features()

ValueError: No features specified for retrieval

In [ ]:
historical_file = RESULTS_DIR / "historical_features.csv"

historical_features.to_csv(
    historical_file,
    index=False
)

print("Saved:")
print(historical_file)

Saved:
/content/skill_gap_feast/results/historical_features.csv


In [ ]:
model_features = [
    "age",
    "programming_skill",
    "python_experience_years",
    "math_skill",
    "ml_knowledge",
    "ai_project_count",
    "online_courses_completed",
    "coding_hours_per_week",
    "communication_skill",
    "problem_solving",
    "certification_flag",
    "internship_flag",
    "technical_skill_score",
    "experience_score",
    "learning_engagement_score",
    "soft_skill_score",
    "industry_readiness_score"
]

X = historical_features[model_features]

y = historical_features["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (500, 17)
y shape: (500,)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 400
Testing samples: 100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Model training completed.")

Model training completed.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Model Accuracy:", accuracy)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Low",
            "Medium",
            "High"
        ]
    )
)

Model Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         Low       1.00      1.00      1.00        32
      Medium       1.00      1.00      1.00        33
        High       1.00      1.00      1.00        35

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print(
    "Latest event timestamp:",
    features["event_timestamp"].max()
)

Latest event timestamp: 2026-05-15 00:00:00+00:00


In [ ]:
%cd /content/skill_gap_feast/feature_repo

!feast materialize-incremental 2026-08-18T23:00:00

%cd /content/skill_gap_feast

/content/skill_gap_feast/feature_repo
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-

In [ ]:
online_features = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {
            "student_id": 150
        }
    ]
).to_dict()

online_features

{'student_id': [150],
 'communication_skill': [9],
 'problem_solving': [10],
 'math_skill': [4],
 'ai_project_count': [0],
 'python_experience_years': [1],
 'industry_readiness_score': [3.1500000953674316],
 'technical_skill_score': [2.3333332538604736],
 'ml_knowledge': [2],
 'internship_flag': [0],
 'experience_score': [1.0],
 'age': [19],
 'certification_flag': [0],
 'soft_skill_score': [9.5],
 'online_courses_completed': [2],
 'coding_hours_per_week': [2],
 'learning_engagement_score': [0.9166666865348816],
 'programming_skill': [1]}

In [ ]:
online_features = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {
            "student_id": 150
        }
    ]
).to_dict()

online_features

{'student_id': [150],
 'communication_skill': [9],
 'problem_solving': [10],
 'math_skill': [4],
 'ai_project_count': [0],
 'python_experience_years': [1],
 'industry_readiness_score': [3.1500000953674316],
 'technical_skill_score': [2.3333332538604736],
 'ml_knowledge': [2],
 'internship_flag': [0],
 'experience_score': [1.0],
 'age': [19],
 'certification_flag': [0],
 'soft_skill_score': [9.5],
 'online_courses_completed': [2],
 'coding_hours_per_week': [2],
 'learning_engagement_score': [0.9166666865348816],
 'programming_skill': [1]}

In [ ]:
online_df = pd.DataFrame(online_features)

display(online_df)

,student_id,communication_skill,problem_solving,math_skill,ai_project_count,python_experience_years,industry_readiness_score,technical_skill_score,ml_knowledge,internship_flag,experience_score,age,certification_flag,soft_skill_score,online_courses_completed,coding_hours_per_week,learning_engagement_score,programming_skill
0,150,9,10,4,0,1,3.15,2.333333,2,0,1.0,19,0,9.5,2,2,0.916667,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_model_input = online_df[
    model_features
]

display(online_model_input)

,age,programming_skill,python_experience_years,math_skill,ml_knowledge,ai_project_count,online_courses_completed,coding_hours_per_week,communication_skill,problem_solving,certification_flag,internship_flag,technical_skill_score,experience_score,learning_engagement_score,soft_skill_score,industry_readiness_score
0,19,1,1,4,2,0,2,2,9,10,0,0,2.333333,1.0,0.916667,9.5,3.15


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
prediction = model.predict(
    online_model_input
)[0]

label_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

predicted_label = label_map[prediction]

print("Student ID: 150")
print("Predicted Skill Gap:", predicted_label)

Student ID: 150
Predicted Skill Gap: High


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_output_file = RESULTS_DIR / "online_features.csv"

online_df.to_csv(
    online_output_file,
    index=False
)

print("Saved:")
print(online_output_file)

Saved:
/content/skill_gap_feast/results/online_features.csv


In [ ]:
online_output_file = RESULTS_DIR / "online_features.csv"

online_df.to_csv(
    online_output_file,
    index=False
)

print("Saved:")
print(online_output_file)

Saved:
/content/skill_gap_feast/results/online_features.csv


In [ ]:
prediction_file = RESULTS_DIR / "prediction.txt"

with open(prediction_file, "w") as f:

    f.write("Curriculum-Industry Skill Gap Prediction\n")
    f.write("----------------------------------------\n")
    f.write("Student ID: 150\n")
    f.write(f"Predicted Skill Gap: {predicted_label}\n")
    f.write(f"Encoded Prediction: {prediction}\n")
    f.write(f"Model Accuracy: {accuracy:.4f}\n")

print(prediction_file.read_text())

Curriculum-Industry Skill Gap Prediction
----------------------------------------
Student ID: 150
Predicted Skill Gap: High
Encoded Prediction: 2
Model Accuracy: 1.0000



In [ ]:
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print()

print("Dataset:")
print("Records:", len(df))
print("Original Columns:", len(df.columns))

print()

print("Feast:")
print("Entity: student")
print("FeatureView: skill_gap_features")
print("Number of features:", len(feature_refs))

print()

print("Machine Learning:")
print(f"Accuracy: {accuracy:.4f}")

print()

print("Online Prediction:")
print("Student ID: 150")
print("Prediction:", predicted_label)

print("=" * 60)

FINAL RESULTS

Dataset:
Records: 500
Original Columns: 16

Feast:
Entity: student
FeatureView: skill_gap_features
Number of features: 17

Machine Learning:
Accuracy: 1.0000

Online Prediction:
Student ID: 150
Prediction: High
